# 06 - Held-out test evaluation

This notebook runs the held-out test split only after notebook 05 has written a freeze record from the full validation run. It checks the checkpoint, preprocessing, thresholds, and validation report hash before scoring. Test labels are never used to choose thresholds or preprocessing.

Most labels in this run are source-derived proxies, not reviewed game labels. This prepared test split already has a test-v1 result for an older checkpoint, so this run is a repeat on an exposed split, not a fresh untouched test. The charts report proxy-label agreement; precision, recall and F1 use the frozen 0.5 reference threshold.

## Prerequisites

Before this run, the following files must exist:

- The selected Referee checkpoint.
- Prepared `test.jsonl` and its images under the same data profile used for validation.
- `freeze.json` and `report.md` from notebook 05, created from the full validation run for this checkpoint and prepared dataset.

This notebook checks each file below. It does not download or prepare test data. This run uses a new output directory so it preserves the earlier test-v1 result. It refuses to overwrite non-empty output.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from referee_evaluation import notebook_tools

REPOSITORY_ROOT = Path.cwd().resolve()
while not (REPOSITORY_ROOT / 'models' / 'referee' / 'recognition').is_dir():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
MODEL_PROJECT = REPOSITORY_ROOT / 'models' / 'referee'


In [ ]:
RUN_HELDOUT_EVALUATION = True
PROFILE = 'train'
CHECKPOINT = MODEL_PROJECT / 'checkpoints' / '20261111T185421Z-7c09d4f3bc' / 'weights' / 'best-04.ckpt'
PREPARED_DIR = MODEL_PROJECT / 'data' / '.local' / 'data' / PROFILE / 'prepared'
VALIDATION_ROOT = MODEL_PROJECT / 'checkpoints' / '20261111T185421Z-7c09d4f3bc' / 'validation'
FREEZE_RECORD = VALIDATION_ROOT / 'freeze.json'
VALIDATION_REPORT = VALIDATION_ROOT / 'report.md'
OUTPUT_ROOT = MODEL_PROJECT / 'checkpoints' / '20261111T185421Z-7c09d4f3bc' / 'test'
BATCH_SIZE = 32
LIMIT = None
OVERWRITE = False

TEST_MANIFEST = PREPARED_DIR / 'manifests' / 'test.jsonl'
PREREQUISITES = {
    'Checkpoint': CHECKPOINT,
    'Prepared test manifest': TEST_MANIFEST,
    'Validation freeze record': FREEZE_RECORD,
    'Validation report': VALIDATION_REPORT,
}
READY = all(path.is_file() for path in PREREQUISITES.values())
for name, path in PREREQUISITES.items():
    print(f"{name}: {'OK' if path.is_file() else 'MISSING'} - {path}")
RESULT_PATH = OUTPUT_ROOT / 'summary.json'
RESULT_EXISTS = RESULT_PATH.is_file()
print('Held-out run enabled:', RUN_HELDOUT_EVALUATION)
print('Checkpoint:', CHECKPOINT)
print('Test manifest:', TEST_MANIFEST)
print('Freeze record:', FREEZE_RECORD)
print('Validation report:', VALIDATION_REPORT)
print('All prerequisites met:', READY)
print('Existing held-out result:', RESULT_PATH if RESULT_EXISTS else 'none')

In [ ]:
MODEL_ENV = dict(os.environ)
MODEL_ENV.pop('PYTHONPATH', None)
MODEL_ENV.pop('PYTHONHOME', None)

if not RUN_HELDOUT_EVALUATION:
    print('Held-out evaluation disabled. Review the prerequisites, then enable it.')
elif not READY:
    raise FileNotFoundError('Resolve the missing prerequisites listed above before running held-out evaluation.')
elif RESULT_EXISTS and not OVERWRITE:
    print('Using the existing held-out result; set OVERWRITE = True only to replace it.')
else:
    command = [
        'uv', 'run', '--project', str(MODEL_PROJECT), '--frozen', '--no-sync',
        'python', '-m', 'recognition.evaluate',
        '--checkpoint', str(CHECKPOINT),
        '--prepared-dir', str(PREPARED_DIR),
        '--split', 'test',
        '--output-dir', str(OUTPUT_ROOT),
        '--preprocessing', 'auto',
        '--freeze-record', str(FREEZE_RECORD),
        '--validation-report', str(VALIDATION_REPORT),
        '--batch-size', str(BATCH_SIZE),
    ]
    if LIMIT is not None:
        command.extend(['--limit', str(LIMIT)])
    if OVERWRITE:
        command.append('--overwrite')
    print('Launching:', ' '.join(command))
    if not notebook_tools.run_interruptible_command(command, cwd=MODEL_PROJECT, env=MODEL_ENV):
        raise RuntimeError('Held-out evaluation interrupted.')

In [ ]:
SUMMARY_PATH = OUTPUT_ROOT / 'summary.json'
if SUMMARY_PATH.is_file():
    summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
    rows = []
    for name in ('text', 'number'):
        channel = summary['channels'][name]
        rows.append({
            'channel': name,
            'encoded_ap': channel['encodedAp'],
            'known_ap': channel.get('knownAp'),
            'reviewed_ap': channel.get('reviewedAp'),
            'frozen_threshold': channel.get('frozenThreshold'),
            'proxy_precision_at_frozen_threshold': (channel.get('encodedAtFrozenThreshold') or {}).get('precision'),
            'proxy_recall_at_frozen_threshold': (channel.get('encodedAtFrozenThreshold') or {}).get('recall'),
            'proxy_f1_at_frozen_threshold': (channel.get('encodedAtFrozenThreshold') or {}).get('f1'),
            'known': channel['support'].get('known'),
            'known_negative': channel['support'].get('negative'),
        })
    display(pd.DataFrame(rows).set_index('channel'))
    display(pd.DataFrame([summary['coverage']]).T)
    print('Split:', summary['split'])
    print('Preprocessing:', summary['preprocessing'])
    print('Frozen thresholds:', summary['frozenThresholds'])

    channel_names = ('text', 'number')
    ap_values = [summary['channels'][name]['encodedAp'] for name in channel_names]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ap_bars = axes[0].bar(channel_names, ap_values, color=['#4e79a7', '#f28e2b'])
    axes[0].bar_label(ap_bars, fmt='%.3f', padding=3)
    axes[0].set_ylim(0, 1.08)
    axes[0].set_ylabel('Average precision')
    axes[0].set_title('Ranking vs encoded proxy labels')
    axes[0].grid(axis='y', alpha=0.25)

    metric_names = ('precision', 'recall', 'f1')
    positions = range(2)
    width = 0.24
    for index, metric in enumerate(metric_names):
        values = [summary['channels'][name]['encodedAtFrozenThreshold'][metric] for name in ('text', 'number')]
        bars = axes[1].bar([position + (index - 1) * width for position in positions], values, width, label=metric)
        axes[1].bar_label(bars, fmt='%.2f', padding=2, fontsize=8)
    axes[1].set_xticks(list(positions), ('text', 'number'))
    axes[1].set_ylim(0, 1.08)
    axes[1].set_ylabel('Score at frozen threshold 0.5')
    axes[1].set_title('Decisions vs encoded proxy labels')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()
    print('These are encoded/source-label proxy metrics, not reviewed or gameplay accuracy. Do not tune from test results.')
else:
    print('No held-out result yet.')